# Índice Invertido

**Objetivo:** Construir un índice invertido sobre el corpus de documentos para permitir búsquedas eficientes por palabra.

## ¿Qué es un índice invertido?

Un **índice invertido** es una estructura de datos que mapea cada término del vocabulario a la lista de documentos en los que aparece:

```
{
  "amor":   ["libro1.txt", "libro3.txt"],
  "ciudad": ["libro2.txt"],
  ...
}
```

Es la estructura fundamental de los motores de búsqueda (Google, Elasticsearch, Lucene, Solr).

| Enfoque | Búsqueda | Construcción |
|---|---|---|
| Búsqueda lineal | O(n · m) por consulta | Inmediata |
| Índice invertido | **O(1)** por consulta | O(n · m) una sola vez |

El índice se construye **una vez** recorriendo todos los documentos, y luego cada búsqueda es una simple consulta al diccionario.

### 1. Listado del corpus

Obtiene la lista de todos los archivos disponibles en el directorio de datos con `os.listdir()`.

- `files` es una lista de strings con los nombres de archivo (sin ruta).
- `len(files)` muestra cuántos documentos componen el corpus — este número determina el tamaño máximo del índice.
- El orden no está garantizado (`os.listdir` devuelve orden del sistema de archivos).

In [3]:
import os
path = '../data/gutenberg/data/'
files = os.listdir(path)
len(files)

1022

### 2. Prototipo comentado — exploración inicial

Código de exploración que fue dejado comentado. Muestra el razonamiento previo a la implementación real:

1. Leer un único archivo.
2. Extraer tokens con `split()` y obtener vocabulario con `set()`.
3. Construir el índice solo para ese archivo con `indice_invertido[palabra] = [archivo]`.

**Problema de este enfoque:** sobreescribe la lista en lugar de hacer `append`, por lo que no acumula múltiples documentos para la misma palabra. La siguiente celda corrige esto.

In [4]:
# contenidos = {}
# archivo = files[0,1]
# indice_invertido = {}

# with open(path + '/' + archivo, 'r', encoding='utf-8') as f:
#     contenido = f.read()
#     print(contenido[:100])
#     palabras = contenido.split()
#     vocabulario = set(palabras)
#     print(palabras[:100])
#     print(len(palabras))
#     print(len(vocabulario))

# for palabra in vocabulario:
#     indice_invertido[palabra] = [archivo]

### 3. Primera implementación funcional (2 archivos)

Construye el índice invertido sobre los **dos primeros archivos** del corpus. Proceso por cada archivo:

1. **Leer** el contenido completo del archivo.
2. **Tokenizar** con `split()` — divide por espacios en blanco.
3. **Vocabulario** con `set()` — elimina duplicados, quedando solo palabras únicas del documento.
4. **Actualizar índice:**
   - Si la palabra **ya existe** en `indice_invertido` → añade el archivo con `.append()`.
   - Si la palabra **no existe** → crea una nueva entrada `[archivo]`.
5. **Imprime** el número de palabras únicas del archivo y el tamaño acumulado del índice.

> **Limitación:** sin limpieza de texto. `"amor,"`, `"amor."` y `"amor"` se indexan como tres términos distintos, fragmentando el vocabulario innecesariamente.

In [5]:
contenidos = {}
indice_invertido = {}

for archivo in files[:2]:
    ''' Se iteran los archivos'''
    with open(path + '/' + archivo, 'r', encoding='utf-8') as f:
        contenido = f.read()
        palabras = contenido.split()
        vocabulario = set(palabras)
        for palabra in vocabulario:
            if palabra in indice_invertido:
                indice_invertido[palabra].append(archivo)
            else:
                indice_invertido[palabra] = [archivo]
        print('Archivo ', archivo + '- Numero de palabras unicas', len(vocabulario))
        print('Numero de palabras del indice invertido', len(indice_invertido))



Archivo  20_poemas_para_ser_leidos_en_el_tranvia.txt- Numero de palabras unicas 1513
Numero de palabras del indice invertido 1513
Archivo  40_years_40_anos_40_ans.txt- Numero de palabras unicas 5152
Numero de palabras del indice invertido 6492


### 4. Inspección del índice (2 archivos)

Imprime el diccionario completo del índice construido sobre los 2 primeros archivos. Permite verificar visualmente la estructura:

```
{ "palabra": ["doc1.txt"], "otra": ["doc1.txt", "doc2.txt"], ... }
```

Las palabras que aparecen en ambos archivos tendrán listas de longitud 2; las que solo aparecen en uno, longitud 1.

In [6]:
print(indice_invertido)

{'“campanile”': ['20_poemas_para_ser_leidos_en_el_tranvia.txt'], 'zapatos.': ['20_poemas_para_ser_leidos_en_el_tranvia.txt'], 'alguien': ['20_poemas_para_ser_leidos_en_el_tranvia.txt'], 'terrazas,': ['20_poemas_para_ser_leidos_en_el_tranvia.txt'], 'lenguas': ['20_poemas_para_ser_leidos_en_el_tranvia.txt', '40_years_40_anos_40_ans.txt'], 'cuatro': ['20_poemas_para_ser_leidos_en_el_tranvia.txt', '40_years_40_anos_40_ans.txt'], 'corcovo': ['20_poemas_para_ser_leidos_en_el_tranvia.txt'], 'cama': ['20_poemas_para_ser_leidos_en_el_tranvia.txt'], '¡Ventanas': ['20_poemas_para_ser_leidos_en_el_tranvia.txt'], 'Uno': ['20_poemas_para_ser_leidos_en_el_tranvia.txt'], 'espera,': ['20_poemas_para_ser_leidos_en_el_tranvia.txt'], 'cabelleras': ['20_poemas_para_ser_leidos_en_el_tranvia.txt'], 'fornican': ['20_poemas_para_ser_leidos_en_el_tranvia.txt'], 'menos': ['20_poemas_para_ser_leidos_en_el_tranvia.txt'], 'Negras': ['20_poemas_para_ser_leidos_en_el_tranvia.txt'], '20': ['20_poemas_para_ser_leidos_e

### 5. Índice invertido mejorado — corpus completo con normalización

Versión mejorada que procesa **todos los archivos** e incorpora limpieza de tokens con expresiones regulares. Pasos añadidos respecto a la versión anterior:

**Limpieza con regex:**
```python
re.sub(r"[^A-Za-zÁÉÍÓÚÜÑáéíóúüñ0-9]", "", p).lower()
```
- Elimina todo carácter que **no sea** letra (español/inglés) o dígito — se van signos de puntuación, guiones, corchetes, etc.
- `.lower()` convierte a minúsculas para unificar `"Amor"`, `"AMOR"` y `"amor"` como el mismo término.

**Filtrado de vacíos:**
```python
set(p for p in palabras_limpias if p)
```
- Descarta strings vacíos `""` que quedan cuando un token era solo puntuación (ej: `"---"` → `""`).

**Resultado:** vocabulario más limpio y compacto → menos entradas en el índice, búsquedas más precisas.

In [7]:
import re

# Diccionario para guardar el índice invertido.
contenidos = {}
indice_invertido = {}

# Recorre cada archivo del corpus.
for archivo in files:
    # Lee el contenido completo del archivo.
    with open(path + "" + archivo, 'r', encoding='utf-8') as f:
        contenido = f.read()
        # Separa el texto en tokens por espacios.
        palabras = contenido.split()
        # Limpia símbolos de cada token (por ejemplo: _[feeling -> feeling).
        palabras_limpias = [re.sub(r"[^A-Za-zÁÉÍÓÚÜÑáéíóúüñ0-9]", "", p).lower() for p in palabras]
        # Elimina tokens vacíos y deja palabras únicas por documento.
        vocabulario = set(p for p in palabras_limpias if p)
        for palabra in vocabulario:
            if palabra in indice_invertido:
                indice_invertido[palabra].append(archivo)
            else:
                indice_invertido[palabra] = [archivo]

        print ('Archivo: ', archivo + ' - Número de palabras únicas: ', len(vocabulario))
        print ('Numero de palabras del indice invertido: ', len(indice_invertido))

Archivo:  20_poemas_para_ser_leidos_en_el_tranvia.txt - Número de palabras únicas:  1322
Numero de palabras del indice invertido:  1322
Archivo:  40_years_40_anos_40_ans.txt - Número de palabras únicas:  3843
Numero de palabras del indice invertido:  4979
Archivo:  7_de_julio.txt - Número de palabras únicas:  8610
Numero de palabras del indice invertido:  12671
Archivo:  abel_sanchez_una_historia_de_pasion.txt - Número de palabras únicas:  4925
Numero de palabras del indice invertido:  15350
Archivo:  actas_capitulares_desde_el_21_hasta_el_25_de_mayo_de_1810_en_buenos_aires.txt - Número de palabras únicas:  3380
Numero de palabras del indice invertido:  17303
Archivo:  adan_y_eva_en_el_paraiso.txt - Número de palabras únicas:  12566
Numero de palabras del indice invertido:  25209
Archivo:  adhesiones_a_la_venta_de_los_ferrocarriles_de_la_provincia.txt - Número de palabras únicas:  8477
Numero de palabras del indice invertido:  30870
Archivo:  adriana_zumaran_novela.txt - Número de pala

### 6. Inspección del índice completo

Imprime el diccionario completo del índice invertido construido sobre todos los archivos del corpus. Para un corpus de ~1000 documentos la salida es muy extensa. La siguiente celda presenta los mismos datos en formato tabular más manejable.

In [ ]:
# Imprime el diccionario completo del índice invertido.
print (indice_invertido)

### 7. Visualización del índice como DataFrame

Convierte el índice invertido en un `DataFrame` de pandas para facilitar la exploración:

```python
pd.DataFrame(list(indice_invertido.items()), columns=['Palabra', 'Archivos'])
```

- `indice_invertido.items()` genera pares `(palabra, lista_de_archivos)`.
- `list(...)` materializa el iterador para pasarlo al constructor del DataFrame.
- Resultado: tabla con dos columnas — **Palabra** y **Archivos** (lista de documentos).

Ventajas sobre `print(indice_invertido)`: permite ordenar por columna, filtrar, y analizar estadísticas (ej: cuántos documentos contienen cada palabra).

In [ ]:
import pandas as pd

# Convierte el índice invertido en una tabla con dos columnas.
df = pd.DataFrame(list(indice_invertido.items()), columns=['Palabra', 'Archivos'])

# Muestra el DataFrame.
df

### 8. Función de búsqueda sobre el índice

Define `buscar(query)` que consulta el índice invertido:

```python
indice_invertido.get(query, [])
```

- `dict.get(key, default)` devuelve el valor si la clave existe, o `default` si no.
- Si `query` está en el índice → retorna la lista de archivos donde aparece.
- Si `query` no está → retorna `[]` (lista vacía) sin lanzar error.

**Complejidad: O(1)** — un solo lookup en el diccionario, independiente del tamaño del corpus. Contrasta con la búsqueda lineal del ejercicio 01 que era O(n · m).

In [ ]:
# Busca una palabra en el índice invertido y devuelve la lista de archivos.
def buscar (query):
    return indice_invertido.get(query, [])

### 9. Prueba de la función de búsqueda

Itera sobre todas las palabras del índice y llama a `buscar()` para cada una.

- Verifica que la función devuelve resultados para cada término indexado.
- En uso real se llamaría con una única consulta del usuario: `buscar("amor")`.
- La salida muestra la lista de archivos por cada palabra — permite inspeccionar cuáles palabras aparecen en muchos documentos (palabras frecuentes) vs. pocas (términos raros).

In [ ]:
# Recorre todas las palabras del índice y muestra sus resultados de búsqueda.
for palabra in indice_invertido.keys():
    print (buscar(palabra))